# LFM Instance Segmentation Example Workflow
This notebook is an example workflow of doing instance segmentation on visual light (RGB) bands of Lunar data. 

You can get started with this notebook by downloading it with:

```bash
wget https://raw.githubusercontent.com/nasa-nccs-hpda/lfm/refs/heads/main/notebooks/finetune_dinov3.ipynb
```

**See the README in the [repo](https://github.com/nasa-nccs-hpda/lfm)** for more info on how to use this notebook, and more on the process of training the model. 

## Imports, Dino Repo Clone

### Grace hopper setup

In [ ]:
# !rm -rf ~/.local/lib

In [ ]:
# !pip install torchmetrics
# !pip install --upgrade transformers
# !pip install rasterio

### Imports etc

In [ ]:
import os
import sys
import torch
import subprocess
from glob import glob

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from transformers import AutoImageProcessor

%matplotlib inline

In [ ]:
repo_dir = "lfm"

# if not os.path.exists(repo_dir):
#     subprocess.run(["git", "clone", f"https://github.com/nasa-nccs-hpda/{repo_dir}"])
# else:
#     subprocess.run(["git", "-C", repo_dir, "pull"])
# subprocess.run(["git", "-C", repo_dir, "checkout", "develop"])
# subprocess.run(["git", "-C", repo_dir, "pull"])

In [ ]:
sys.path.append("lfm")

from lfm.tasks.inst_segmentation.iseg_dataset import get_dataloaders, calculate_dataset_statistics
from lfm.tasks.inst_segmentation.iseg_driver import train_model
from lfm.tasks.inst_segmentation.iseg_mask2former_model import create_mask2former_dinov3_model

## Main workflow

1. Define user-configured variables
2. Create dataloaders from files on /explore/nobackup/.
3. Load DinoV3 encoder, create encoder/decoder finetuning model.
4. Train model, print training stats, and visualize results. 

### User Config

In [ ]:
# Local image of DinoV3 encoder
weights_local_checkpoint = '/explore/nobackup/projects/ilab/models/dinov3/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'

# Data paths
INPUT_ROOT_DIR = "/explore/nobackup/projects/lfm/model_inputs/300_300_inputs/7_band_vis_uv/inst_seg"
INPUT_DIR = f"{INPUT_ROOT_DIR}/chips"  # Where inputs are stored
LABEL_DIR = f"{INPUT_ROOT_DIR}/labels_npy"  # Where labels are stored
INPUT_FILE_TYPE = ".tif"
LABEL_FILE_TYPE = ".npz"

# Output dir (this will be created automatically)
OUTPUT_DIR = "./outputs/inst_seg_7_band_flex_50ep"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

# Dataset parameters
MAX_SAMPLES = 500  # Set to None to use all samples, or an integer to limit
TRAIN_SPLIT = 0.8  # 80% train, 20% validation

# Training hyperparameters
BATCH_SIZE = 16  # Number of images fed into the model at a time
NUM_EPOCHS = 50  # 10 is the default value, increase for more learning
LEARNING_RATE = 5e-5  # Starting learning rate
WEIGHT_DECAY = 1e-3  # Weight decay of optimizer
NUM_WORKERS = 8  # Used for parallelization
WARMUP_EPOCHS = 10  # Number of epochs for warmup LR scheduler
PATIENCE = 50  # How many epochs wait before early stopping

# Model parameters
TARGET_SIZE = (304, 304)  # Input size for DINO model
FREEZE_ENCODER = False
NUM_BANDS = 7  # Number of bands to use on the model
BAND_FILTER = None  # Bands to keep, in order of filtering

# Visualization and saving
CHECKPOINT_EVERY = 10  # Save checkpoint every N epochs
VISUALIZE_EVERY = 10  # Create visualizations every N epochs

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

### Create dataloaders

#### Calculate dataset stats to normalize items

In [ ]:
print("\n" + "="*60)
print("STEP 1: Computing dataset statistics.")
print("="*60)

# Check if statistics already exist
mean_path = os.path.join(OUTPUT_DIR, "dataset_mean.npy")
std_path = os.path.join(OUTPUT_DIR, "dataset_std.npy")

if os.path.exists(mean_path) and os.path.exists(std_path):
    print("Loading existing dataset statistics...")
    MEAN = np.load(mean_path)
    STD = np.load(std_path)
    print(f"Mean per channel: {MEAN}")
    print(f"Std per channel: {STD}")
else:
    print("Computing dataset statistics...")
    MEAN, STD = calculate_dataset_statistics(INPUT_DIR, INPUT_FILE_TYPE)

    # Save statistics
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    np.save(mean_path, MEAN)
    np.save(std_path, STD)
    print(f"✓ Saved statistics to {OUTPUT_DIR}")

#### Create image processor (will handle normalization)

In [ ]:
print("\nSTEP 2: Creating image processor.")
print("="*60)
BASE_MODEL = "facebook/mask2former-swin-large-coco-instance"

print("\nCreating image processor with custom normalization...")
print(f"  Base model: {BASE_MODEL}")
print(f"  Target size: {TARGET_SIZE}")
print(f"  Custom mean: {MEAN.tolist()}")
print(f"  Custom std: {STD.tolist()}")

# Create image processor with your dataset's statistics
image_processor = AutoImageProcessor.from_pretrained(
    BASE_MODEL,
    do_resize=True,
    size={"height": TARGET_SIZE[0], "width": TARGET_SIZE[1]},
    do_normalize=False,
    do_reduce_labels=False,     # Keep background as 0
    data_format="channels_first",  # ADD THIS: Explicitly set output format
    input_data_format="channels_first",  # ADD THIS: Explicitly set input format
)

#### Create dataloaders

In [ ]:
# Now create dataloaders with image_processor
print("\nSTEP 3: Creating dataloaders.")
print("="*60)

train_loader, val_loader, _, _ = get_dataloaders(
    image_dir=INPUT_DIR,
    label_dir=LABEL_DIR,
    image_processor=image_processor,
    batch_size=BATCH_SIZE,
    train_split=TRAIN_SPLIT,
    num_workers=NUM_WORKERS,
    target_size=TARGET_SIZE,
    max_samples=MAX_SAMPLES,
    seed=42,
    stats_save_dir=OUTPUT_DIR,
    input_file_type=INPUT_FILE_TYPE,
    label_file_type=LABEL_FILE_TYPE,
    band_filter=BAND_FILTER
)

print(f"    Train batches: {len(train_loader)}")
print(f"    Val batches: {len(val_loader)}")

### Load Encoder and Create Model

In [ ]:
# Required for mask2former model
label2id = {
    "background": 0,
    "crater": 1,
}
id2label = {v: k for k, v in label2id.items()}

In [ ]:
print("\n" + "="*60)
print("Loading mask2former Dino model...")
print("="*60)

model = create_mask2former_dinov3_model(
    label2id,
    id2label,
    expected_channels=[192, 384, 768, 1536],
    freeze_backbone=False,
    hub_token=None,
    num_bands=NUM_BANDS
)
model = model.to(device)

### Run Training

In [ ]:
print("\n" + "="*60)
print("Starting training.")
print("="*60)

train_losses, val_losses = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    mode="train",
    output_dir=OUTPUT_DIR,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    checkpoint_every=CHECKPOINT_EVERY,
    visualize_every=VISUALIZE_EVERY,
    image_processor=image_processor,
    device=device,
    warmup_epochs=WARMUP_EPOCHS,
    early_stopping_patience=PATIENCE,
    max_grad_norm=1.0
)

## Display some of the output visualizations

The training of the model is already producing some visualizations every N epochs.
Here we open some of the visualizations to look at them from the notebook.

In [ ]:
visualization_dir = os.path.join(OUTPUT_DIR, 'visualizations')
visualization_filenames = sorted(glob(os.path.join(visualization_dir, '*.png')))

In [ ]:
for vis_filename in visualization_filenames:
    img = mpimg.imread(vis_filename)
    plt.figure(figsize=(16, 14))
    plt.imshow(img)
    plt.axis("off")
    plt.show()